# Notebook 01: Data Loading and Initial Profiling

## Purpose
This notebook initializes Spark, loads raw AFDC station and Census inputs, and performs a first-pass profiling review.

## What this notebook establishes
- Input files can be located and read correctly
- Core schemas and key columns are identified
- Early data quality signals are documented before cleaning begins

In [1]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@17/17.0.19/libexec/openjdk.jdk/Contents/Home'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EV_Charging_Analysis") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 11:14:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [3]:
import os

# Get the current working directory - wherever your notebook is running from
BASE_DIR = os.getcwd()

# Input file paths
AFDC_PATH = os.path.join(BASE_DIR, "alt_fuel_stations.csv")
CENSUS_PATH = os.path.join(BASE_DIR, "acs_zcta_combined.csv")

# Output directory
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Print to verify - check these look correct
print("Base dir:", BASE_DIR)
print("AFDC path:", AFDC_PATH)
print("Census path:", CENSUS_PATH)
print("Output dir:", OUTPUT_DIR)

# Verify the files actually exist before even trying to load them
print("\nAFDC file exists:", os.path.exists(AFDC_PATH))
print("Census file exists:", os.path.exists(CENSUS_PATH))

Base dir: /Users/krishjani/Documents/Sem 2/Big Data/EV Infrastructure Equity Analyzer
AFDC path: /Users/krishjani/Documents/Sem 2/Big Data/EV Infrastructure Equity Analyzer/alt_fuel_stations.csv
Census path: /Users/krishjani/Documents/Sem 2/Big Data/EV Infrastructure Equity Analyzer/acs_zcta_combined.csv
Output dir: /Users/krishjani/Documents/Sem 2/Big Data/EV Infrastructure Equity Analyzer/output

AFDC file exists: True
Census file exists: True


In [4]:
# multiLine + escape helps Spark safely parse quoted fields that span lines
# in the AFDC source export.
df_afdc_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(AFDC_PATH)

print("AFDC row count:", df_afdc_raw.count())
print("AFDC column count:", len(df_afdc_raw.columns))

AFDC row count: 85659
AFDC column count: 75


In [5]:
# This prints every column name and its inferred data type
df_afdc_raw.printSchema()

root
 |-- Fuel Type Code: string (nullable = true)
 |-- Station Name: string (nullable = true)
 |-- Street Address: string (nullable = true)
 |-- Intersection Directions: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- Plus4: string (nullable = true)
 |-- Station Phone: string (nullable = true)
 |-- Status Code: string (nullable = true)
 |-- Expected Date: string (nullable = true)
 |-- Groups With Access Code: string (nullable = true)
 |-- Access Days Time: string (nullable = true)
 |-- Cards Accepted: string (nullable = true)
 |-- BD Blends: string (nullable = true)
 |-- NG Fill Type Code: string (nullable = true)
 |-- NG PSI: string (nullable = true)
 |-- EV Level1 EVSE Num: integer (nullable = true)
 |-- EV Level2 EVSE Num: integer (nullable = true)
 |-- EV DC Fast Count: integer (nullable = true)
 |-- EV Other Info: string (nullable = true)
 |-- EV Network: string (nullable = true)
 |-- EV Net

In [7]:
df_census_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(CENSUS_PATH)

print("Census row count:", df_census_raw.count())
print("Census column count:", len(df_census_raw.columns))

df_census_raw.printSchema()

Census row count: 33772
Census column count: 3
root
 |-- zcta: integer (nullable = true)
 |-- total_population: integer (nullable = true)
 |-- median_household_income: double (nullable = true)



In [8]:
# See the first 5 rows of AFDC
# truncate=False means don't cut off long values
df_afdc_raw.show(5, truncate=False)

+--------------+---------------------------------+------------------+------------------------+-----------+-----+-----+-----+-------------+-----------+-------------+-----------------------+-------------------------------+--------------+---------+-----------------+------+------------------+------------------+----------------+-------------+--------------+------------------------------------------------------------+--------------+----------------+-----------------+-------------------+----+-----------------------+---------------+-----------------+-------------------+----------+--------------------+----------------+-----------+----------------+------------------------+-------+--------------------------------+-------------------------+------------------+--------------------------------+------------------+-----------+------------------+-------------------+--------------+-----------------+----------------------------+------------------------------+--------------------+--------------------------

26/05/03 11:16:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
# See the first 5 rows of Census
df_census_raw.show(5, truncate=False)

+----+----------------+-----------------------+
|zcta|total_population|median_household_income|
+----+----------------+-----------------------+
|601 |16721           |18571.0                |
|602 |37510           |21702.0                |
|603 |48317           |19243.0                |
|606 |5435            |20226.0                |
|610 |25413           |23732.0                |
+----+----------------+-----------------------+
only showing top 5 rows


In [10]:
# Print all column names clearly
for i, col in enumerate(df_afdc_raw.columns):
    print(f"{i+1:3}. {col}")

  1. Fuel Type Code
  2. Station Name
  3. Street Address
  4. Intersection Directions
  5. City
  6. State
  7. ZIP
  8. Plus4
  9. Station Phone
 10. Status Code
 11. Expected Date
 12. Groups With Access Code
 13. Access Days Time
 14. Cards Accepted
 15. BD Blends
 16. NG Fill Type Code
 17. NG PSI
 18. EV Level1 EVSE Num
 19. EV Level2 EVSE Num
 20. EV DC Fast Count
 21. EV Other Info
 22. EV Network
 23. EV Network Web
 24. Geocode Status
 25. Latitude
 26. Longitude
 27. Date Last Confirmed
 28. ID
 29. Updated At
 30. Owner Type Code
 31. Federal Agency ID
 32. Federal Agency Name
 33. Open Date
 34. Hydrogen Status Link
 35. NG Vehicle Class
 36. LPG Primary
 37. E85 Blender Pump
 38. EV Connector Types
 39. Country
 40. Intersection Directions (French)
 41. Access Days Time (French)
 42. BD Blends (French)
 43. Groups With Access Code (French)
 44. Hydrogen Is Retail
 45. Access Code
 46. Access Detail Code
 47. Federal Agency Code
 48. Facility Type
 49. CNG Dispenser Num
 5

In [11]:
# These are the only columns relevant to our analysis
key_columns = [
    "ID",
    "Station Name",
    "City",
    "State",
    "ZIP",
    "Latitude",
    "Longitude",
    "Access Code",
    "Status Code",
    "Open Date",
    "EV Level1 EVSE Num",
    "EV Level2 EVSE Num",
    "EV DC Fast Count",
    "EV Network",
    "EV Connector Types"
]

df_afdc_subset = df_afdc_raw.select(key_columns)
print("Subset shape:", df_afdc_subset.count(), "rows,", len(df_afdc_subset.columns), "columns")

Subset shape: 85659 rows, 15 columns


## Data Quality and Distribution Checks

This section quantifies nulls, categorical distributions, coordinate ranges, and formatting issues.
These diagnostics guide the cleaning logic implemented in Notebook 02.

In [12]:
from pyspark.sql import functions as F

# Cache total row count once because per-column null scans are already expensive.
total = df_afdc_subset.count()

print(f"Total rows: {total}\n")
print(f"{'Column':<25} {'Null Count':>12} {'Null %':>10}")
print("-" * 50)

for col_name in df_afdc_subset.columns:
    null_count = df_afdc_subset.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / total) * 100
    print(f"{col_name:<25} {null_count:>12,} {null_pct:>9.1f}%")

Total rows: 85659

Column                      Null Count     Null %
--------------------------------------------------
ID                                   0       0.0%
Station Name                         0       0.0%
City                                 6       0.0%
State                                0       0.0%
ZIP                                  6       0.0%
Latitude                             0       0.0%
Longitude                            0       0.0%
Access Code                          0       0.0%
Status Code                          0       0.0%
Open Date                          531       0.6%
EV Level1 EVSE Num              84,977      99.2%
EV Level2 EVSE Num              14,116      16.5%
EV DC Fast Count                70,451      82.2%
EV Network                          15       0.0%
EV Connector Types                  19       0.0%


In [13]:
print("=== Access Code Values ===")
df_afdc_subset.groupBy("Access Code").count().orderBy("count", ascending=False).show()

print("=== Status Code Values ===")
df_afdc_subset.groupBy("Status Code").count().orderBy("count", ascending=False).show()

print("=== EV Network (top 20) ===")
df_afdc_subset.groupBy("EV Network").count().orderBy("count", ascending=False).show(20)

=== Access Code Values ===
+-----------+-----+
|Access Code|count|
+-----------+-----+
|     public|79964|
|    private| 5695|
+-----------+-----+

=== Status Code Values ===
+-----------+-----+
|Status Code|count|
+-----------+-----+
|          E|83650|
|          T| 1607|
|          P|  402|
+-----------+-----+

=== EV Network (top 20) ===
+-------------------+-----+
|         EV Network|count|
+-------------------+-----+
|ChargePoint Network|45267|
|      Non-Networked| 9251|
|      Blink Network| 6109|
|  Tesla Destination| 5321|
|              Tesla| 3016|
|         EV Connect| 1648|
|              AMPUP| 1389|
|       eVgo Network| 1227|
|                FLO| 1206|
|  Electrify America| 1150|
|               LOOP|  992|
|              RED_E|  841|
|            VIALYNK|  812|
|              SWTCH|  647|
|          OpConnect|  495|
|          EVGATEWAY|  447|
|        FORD_CHARGE|  344|
|             NOODOE|  331|
|           CHARGEUP|  313|
|          UNIVERSAL|  303|
+-----------

In [14]:
print("=== Latitude Stats ===")
df_afdc_subset.select(
    F.min("Latitude").alias("min_lat"),
    F.max("Latitude").alias("max_lat"),
    F.count(F.when(F.col("Latitude").isNull(), 1)).alias("null_lat")
).show()

print("=== Longitude Stats ===")
df_afdc_subset.select(
    F.min("Longitude").alias("min_lon"),
    F.max("Longitude").alias("max_lon"),
    F.count(F.when(F.col("Longitude").isNull(), 1)).alias("null_lon")
).show()

# Check for coordinates outside valid US range
# US lat: roughly 24 to 50, lon: roughly -125 to -66
print("=== Rows outside US coordinate range ===")
df_afdc_subset.filter(
    (F.col("Latitude") < 18) | (F.col("Latitude") > 72) |
    (F.col("Longitude") < -180) | (F.col("Longitude") > -60)
).count()

=== Latitude Stats ===
+-----------+---------+--------+
|    min_lat|  max_lat|null_lat|
+-----------+---------+--------+
|17.99580829|64.852466|       0|
+-----------+---------+--------+

=== Longitude Stats ===
+-----------+------------+--------+
|    min_lon|     max_lon|null_lon|
+-----------+------------+--------+
|-176.636362|-65.65084245|       0|
+-----------+------------+--------+

=== Rows outside US coordinate range ===


1

In [15]:
print("=== Sample Open Date values ===")
df_afdc_subset.select("Open Date") \
    .filter(F.col("Open Date").isNotNull()) \
    .show(20, truncate=False)

=== Sample Open Date values ===
+----------+
|Open Date |
+----------+
|1999-10-15|
|1995-08-30|
|1999-10-15|
|2018-05-01|
|1999-10-15|
|2016-01-01|
|1999-10-15|
|2019-04-01|
|1999-10-15|
|1997-07-30|
|2012-12-11|
|1997-08-30|
|2014-01-20|
|1997-08-30|
|1997-08-30|
|1997-08-31|
|2012-05-04|
|1998-06-30|
|1998-05-30|
|2011-12-12|
+----------+
only showing top 20 rows


In [16]:
print("=== Sample ZIP values ===")
df_afdc_subset.select("ZIP").show(20)

print("=== ZIP length distribution ===")
df_afdc_subset.withColumn("zip_length", F.length(F.col("ZIP").cast("string"))) \
    .groupBy("zip_length").count() \
    .orderBy("zip_length") \
    .show()

=== Sample ZIP values ===
+-----+
|  ZIP|
+-----+
|91352|
|90015|
|90012|
|90803|
|90744|
|91342|
|90012|
|90016|
|90013|
|92037|
|91343|
|92503|
|91103|
|91105|
|91105|
|91030|
|90802|
|90045|
|93030|
|92262|
+-----+
only showing top 20 rows
=== ZIP length distribution ===
+----------+-----+
|zip_length|count|
+----------+-----+
|      NULL|    6|
|         2|    1|
|         3|   12|
|         4|  137|
|         5|85503|
+----------+-----+



In [17]:
# Check the one row outside coordinate range
print("=== Row outside US coordinate range ===")
df_afdc_subset.filter(
    (F.col("Latitude") < 18) | (F.col("Latitude") > 72) |
    (F.col("Longitude") < -180) | (F.col("Longitude") > -60)
).show(truncate=False)

=== Row outside US coordinate range ===
+------+-------------------------+------------+-----+-----+-----------+------------+-----------+-----------+----------+------------------+------------------+----------------+-------------+------------------+
|ID    |Station Name             |City        |State|ZIP  |Latitude   |Longitude   |Access Code|Status Code|Open Date |EV Level1 EVSE Num|EV Level2 EVSE Num|EV DC Fast Count|EV Network   |EV Connector Types|
+------+-------------------------+------------+-----+-----+-----------+------------+-----------+-----------+----------+------------------+------------------+----------------+-------------+------------------+
|399921|McDonald's - Ponce Bypass|Ponce Bypass|PR   |00716|17.99580829|-66.61916287|public     |T          |2023-09-05|NULL              |1                 |NULL            |Blink Network|J1772             |
+------+-------------------------+------------+-----+-----+-----------+------------+-----------+-----------+----------+---------

In [18]:
# Check the Country column from the full dataset
print("=== Country distribution ===")
df_afdc_raw.groupBy("Country").count().orderBy("count", ascending=False).show()

=== Country distribution ===
+-------+-----+
|Country|count|
+-------+-----+
|     US|85659|
+-------+-----+



In [19]:
# Investigate rows where all three charger counts are null simultaneously
print("=== Rows where ALL charger counts are null ===")
all_null_chargers = df_afdc_subset.filter(
    F.col("EV Level1 EVSE Num").isNull() &
    F.col("EV Level2 EVSE Num").isNull() &
    F.col("EV DC Fast Count").isNull()
)
print("Count:", all_null_chargers.count())
all_null_chargers.show(5, truncate=False)

=== Rows where ALL charger counts are null ===
Count: 17
+------+---------------------------------+--------------+-----+-----+---------+-----------+-----------+-----------+----------+------------------+------------------+----------------+-------------+------------------+
|ID    |Station Name                     |City          |State|ZIP  |Latitude |Longitude  |Access Code|Status Code|Open Date |EV Level1 EVSE Num|EV Level2 EVSE Num|EV DC Fast Count|EV Network   |EV Connector Types|
+------+---------------------------------+--------------+-----+-----+---------+-----------+-----------+-----------+----------+------------------+------------------+----------------+-------------+------------------+
|6509  |Bristol Farms                    |South Pasadena|CA   |91030|34.118433|-118.149597|public     |T          |1997-08-31|NULL              |NULL              |NULL            |Non-Networked|NULL              |
|27335 |Vallejo Ferry Terminal           |Vallejo       |CA   |94590|38.105062|-122

In [ ]:
## Notebook 01 Output

At the end of this profiling pass, we have a validated AFDC subset schema and a clear list of data issues to address next:
- null and sparse charger fields
- ZIP normalization requirements
- date parsing readiness
- out-of-scope records to filter during cleaning